# QuantiPhy — Stage 1: GroundingDINO + SAM2 → artifact

Notebook này chỉ chạy detection/segmentation/tracking rồi lưu `tracks.json` và mask `.npz` lên Google Drive. Sau khi hoàn tất, có thể **Disconnect and delete runtime**; Stage 2 sẽ tải artifact mà không chạy lại GroundingDINO/SAM2.


## 0. Mount Drive và cấu hình đường dẫn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# SỬA đường dẫn này nếu project nằm ở thư mục khác.
PROJECT_ROOT = Path('/content/drive/MyDrive/quantiphy_baseline')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/quantiphy_artifacts/stage1')
TEST_VIDEO_ID = 'simulation_0009'  # chứa câu hỏi outer end of the pier

required = [
    PROJECT_ROOT / 'requirements-vision.txt',
    PROJECT_ROOT / 'src/quantiphy_baseline/vision/pipeline.py',
    PROJECT_ROOT / 'data/build_quantiphy_jsonl.py',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Sửa PROJECT_ROOT. Thiếu: ' + ', '.join(missing)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
%cd {PROJECT_ROOT}
print('Project :', PROJECT_ROOT)
print('Artifact:', ARTIFACT_ROOT)


## 1. Cài dependency và kiểm tra T4


In [ ]:
%pip install -q -r requirements-vision.txt
%pip install -q datasets pandas matplotlib huggingface_hub


In [ ]:
import sys, torch, transformers
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('Python      :', sys.version.split()[0])
print('PyTorch     :', torch.__version__)
print('Transformers:', transformers.__version__)
print('GPU         :', torch.cuda.get_device_name(0))
print('VRAM        : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 2**30))


## 2. Chuẩn bị metadata và video


In [ ]:
import json, subprocess
from datasets import load_dataset

grouped_path = PROJECT_ROOT / 'data/processed/grouped_by_video.jsonl'
if not grouped_path.exists():
    dataset = load_dataset('PaulineLi/QuantiPhy-validation')
    split_name = 'validation' if 'validation' in dataset else next(iter(dataset.keys()))
    csv_path = PROJECT_ROOT / 'data/raw/validation_dataset.csv'
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    dataset[split_name].to_pandas().to_csv(csv_path, index=False)
    subprocess.run([
        sys.executable, str(PROJECT_ROOT / 'data/build_quantiphy_jsonl.py'),
        '--input', str(csv_path),
        '--parsed-out', str(PROJECT_ROOT / 'data/processed/parsed_questions.jsonl'),
        '--grouped-out', str(grouped_path),
    ], check=True)

with grouped_path.open(encoding='utf-8') as handle:
    groups = [json.loads(line) for line in handle if line.strip()]
group = next((item for item in groups if item['video_id'] == TEST_VIDEO_ID), None)
assert group is not None, f'Không tìm thấy {TEST_VIDEO_ID}'
for question in group['questions']:
    print(question['qa_id'], '::', question['raw_question'])


In [ ]:
from huggingface_hub import hf_hub_download

video_dir = PROJECT_ROOT / 'validation_videos'
video_dir.mkdir(parents=True, exist_ok=True)
video_path = video_dir / f'{TEST_VIDEO_ID}.mp4'
if not video_path.exists():
    downloaded = hf_hub_download(
        repo_id='PaulineLi/QuantiPhy-validation',
        repo_type='dataset',
        filename=f'validation_videos/{TEST_VIDEO_ID}.mp4',
        local_dir=str(PROJECT_ROOT),
    )
    video_path = Path(downloaded)
assert video_path.exists()
print(video_path, '%.2f MB' % (video_path.stat().st_size / 2**20))


## 3. Kiểm tra plan và parent tracking requests

Stage 1 dùng plan hiện có để biết cần track parent nào. Semantic LLM/VLM mạnh sẽ được kiểm tra riêng trong Stage 2.


In [ ]:
from quantiphy_baseline.measurement_plan import build_measurement_plans
from quantiphy_baseline.vision.entity_specs import build_tracking_requests

plans = build_measurement_plans(group)
requests = build_tracking_requests(group, plans=plans)
for plan in plans:
    print('PLAN', plan.qa_id, plan.planner_version)
    for operand in plan.operands:
        print(' ', operand.role, operand.tracking_key, operand.landmark.kind,
              operand.landmark.candidate_generator)
print('\nTRACK REQUESTS')
for request in requests:
    print(request.entity_key, request.prompts, request.roles, request.preferred_times_s)


## 4. Chạy GroundingDINO + SAM2 và lưu artifact


In [ ]:
from quantiphy_baseline.vision.grounder import GroundingDinoGrounder
from quantiphy_baseline.vision.sam2_tracker import Sam2Tracker
from quantiphy_baseline.vision.pipeline import SegmentTrackPipeline

grounder = GroundingDinoGrounder(
    model_id='IDEA-Research/grounding-dino-tiny',
    threshold=0.28,
    text_threshold=0.22,
)
# GroundingDINO deformable attention ổn định hơn ở FP32 trên T4.
if grounder.dtype != torch.float32:
    grounder.model.to(dtype=torch.float32)
    grounder.dtype = next(grounder.model.parameters()).dtype
assert grounder.dtype == torch.float32

tracker = Sam2Tracker(model_id='facebook/sam2.1-hiera-small')
pipeline = SegmentTrackPipeline(
    grounder=grounder,
    tracker=tracker,
    output_dir=ARTIFACT_ROOT,
    anchor_samples=5,
)
result = pipeline.process_group(group, video_dir=video_dir)
print('Track JSON:', ARTIFACT_ROOT / 'tracks' / f'{TEST_VIDEO_ID}.json')
print('Objects   :', list(result['objects']))


## 5. Kiểm tra artifact và lưu group handoff


In [ ]:
handoff_dir = ARTIFACT_ROOT / 'handoff' / TEST_VIDEO_ID
handoff_dir.mkdir(parents=True, exist_ok=True)
group_path = handoff_dir / 'group.json'
group_path.write_text(json.dumps(group, indent=2), encoding='utf-8')
track_json = ARTIFACT_ROOT / 'tracks' / f'{TEST_VIDEO_ID}.json'
assert track_json.exists()

successful = 0
for key, obj in result['objects'].items():
    if obj.get('error'):
        print('FAILED:', key, obj['error'])
    for instance in obj.get('instances', []):
        mask_path = Path(instance['mask_path'])
        assert mask_path.exists(), mask_path
        successful += 1
        print('OK:', instance['track_id'], mask_path, instance['summary'])
assert successful > 0, 'Không có track nào thành công'
print('\nHANDOFF READY')
print('group     =', group_path)
print('track_json=', track_json)


## 6. Visualize nhanh parent masks


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from quantiphy_baseline.vision.pipeline import load_bitpacked_masks
from quantiphy_baseline.vision.video_io import load_video_pil

frames, decoded_fps = load_video_pil(video_path)
sample_indices = np.unique(np.linspace(0, len(frames) - 1, 4).round().astype(int))
for key, obj in result['objects'].items():
    for instance in obj.get('instances', []):
        masks = load_bitpacked_masks(instance['mask_path']).astype(bool)
        fig, axes = plt.subplots(1, len(sample_indices), figsize=(16, 4))
        for ax, frame_idx in zip(np.atleast_1d(axes), sample_indices):
            image = np.asarray(frames[frame_idx]).copy()
            overlay = image.copy(); overlay[masks[frame_idx]] = (0, 255, 0)
            ax.imshow((0.65 * image + 0.35 * overlay).astype(np.uint8))
            ax.set_title(f'frame {frame_idx}')
            ax.axis('off')
        fig.suptitle(instance['track_id'])
        plt.show()


## Hoàn tất Stage 1

Bây giờ chọn **Runtime → Disconnect and delete runtime**. Mở notebook `colab_stage2_vlm_landmark.ipynb`; notebook đó chỉ tải masks/video/artifact từ Drive và dành VRAM cho VLM.
